In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os

zip_path = "/content/drive/MyDrive/financial_rag_rebuild.zip"

print("ZIP exists:", os.path.exists(zip_path))
print("ZIP size:", os.path.getsize(zip_path), "bytes")

ZIP exists: True
ZIP size: 223131685 bytes


In [3]:
import zipfile
import os
import shutil

zip_path = "/content/drive/MyDrive/financial_rag_rebuild.zip"
project_path = "/content/financial_rag_project"

# Clean any previous extraction
if os.path.exists(project_path):
    shutil.rmtree(project_path)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(project_path)

print("Extracted successfully.")
print(os.listdir(project_path))

Extracted successfully.
['data', 'requirements.txt', 'build_index.py', 'src']


In [4]:
import os

project_path = "/content/financial_rag_project"

print("Project root:", os.listdir(project_path))
print("data/raw exists:", os.path.isdir(os.path.join(project_path, "data", "raw")))
print("src exists:", os.path.isdir(os.path.join(project_path, "src")))
print("build_index.py exists:", os.path.isfile(os.path.join(project_path, "build_index.py")))
print("requirements.txt exists:", os.path.isfile(os.path.join(project_path, "requirements.txt")))

pdfs = list(
    os.path.join(project_path, "data", "raw")
    for _ in [0]
)

print("PDF count:", len([
    f for root, _, files in os.walk(os.path.join(project_path, "data", "raw"))
    for f in files
    if f.lower().endswith(".pdf")
]))

Project root: ['data', 'requirements.txt', 'build_index.py', 'src']
data/raw exists: True
src exists: True
build_index.py exists: True
requirements.txt exists: True
PDF count: 25


In [5]:
%cd /content/financial_rag_project

!pip install -r requirements.txt -q

/content/financial_rag_project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-

In [6]:
from src.chunking import ChunkCreator
from src.document_loader import DocumentLoader

# Verify chunking
chunker = ChunkCreator()

print("Chunk size:", chunker.text_splitter._chunk_size)
print("Chunk overlap:", chunker.text_splitter._chunk_overlap)

# Verify loader path
loader = DocumentLoader()

print("Loader path:", loader.data_dir)
print("PDF count:", len(list(loader.data_dir.rglob("*.pdf"))))

Chunk size: 1500
Chunk overlap: 300
Loader path: data/raw
PDF count: 25


In [7]:
!python build_index.py

Building Financial RAG Knowledge Base

Loading Documents...
Loaded 2840 documents.

Creating Chunks...
Created 8661 chunks.

Loading Embedding Model...
Using embedding device: cpu
modules.json: 100% 349/349 [00:00<00:00, 922kB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 494kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 25.3MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 258kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.61MB/s]

model.safetensors: downloading bytes:  87% 78.6M/90.9M [00:01<00:00, 101MB/s, 6.40MB/s  ] 
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 60.0MB/s, 8.14MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 64.1MB/s, 8.82MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 6013.72it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.14MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 10.2MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 32.8MB/s]
special_tokens_map.json: 100% 112/

In [8]:
from pathlib import Path

vector_db_path = Path("data/vector_db")

print("Vector DB exists:", vector_db_path.exists())
print("Vector DB files:")

for file in vector_db_path.iterdir():
    print(" -", file.name, file.stat().st_size, "bytes")

Vector DB exists: True
Vector DB files:
 - index.faiss 13303341 bytes
 - index.pkl 12115001 bytes


In [9]:
import shutil
from pathlib import Path

source = Path("/content/financial_rag_project/data/vector_db")
destination = Path("/content/drive/MyDrive/financial_rag_vector_db")

# Remove any previous copy
if destination.exists():
    shutil.rmtree(destination)

shutil.copytree(source, destination)

print("Vector DB copied successfully.")
print("Files:")
for f in destination.iterdir():
    print(" -", f.name, f.stat().st_size, "bytes")

Vector DB copied successfully.
Files:
 - index.faiss 13303341 bytes
 - index.pkl 12115001 bytes
